In [ ]:
# run after 6_regressions/regression_analyses.R (results_multilevel_R.csv.
# requires: pandas, numpy, matplotlib

import sys, os, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42   
matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator


CSV_PATH = "results_multilevel_R.csv"
OUTPUT_FOLDER = "figures"

# Reference groups 
REFERENCE = {
    "Model 1": "Politicians",
    "Model 2": "National News",
    "Model 3": "Topic 1: Generic captions/reactions",
}

# model names
MODEL_NAME = {
    "Model 1": "Post Sender",
    "Model 2": "News Type",
    "Model 3": "Post Topic",
}


def model_label(m):
    """Returns, for example, 'Model 1: Post Sender' (or simply 'Model 1' if no name is specified)."""
    return f"{m}: {MODEL_NAME[m]}" if MODEL_NAME.get(m) else m


# Layout - APA-inspired
FIG_WIDTH = 5.15                    
APA_SETTING = {"fig1": {"height": 485, "ms": 22, "lw": 1.3, "step": 0.24},
             "fig2": {"height": 485, "ms": 12, "lw": 1.0, "step": 0.25}}
WRITING_APA = 8.5        
LABEL_WIDTH = 28       
X_EXTRA = 1.1          
FOOTNOTE_IN_FIGURE = False  

OUTCOMES = ["Hate speech", "Explicit hate speech", "Implicit hate speech", "Negative stereotyping"]
SORT_BY = "Hate speech"          
ALL_TICKS = [0.1, 0.2, 0.5, 1, 2, 5, 10, 20]

# Colors and labels
LABELS = {"Hate speech": "Hate speech (overall)"}
COLORS = {"Hate speech": "#222222", "Explicit hate speech": "#D55E00",
          "Implicit hate speech": "#0072B2", "Negative stereotyping": "#009E73"}
MARKERS = {"Hate speech": "D", "Explicit hate speech": "o",
           "Implicit hate speech": "s", "Negative stereotyping": "^"}

APA_RC = {"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
          "font.size": WRITING_APA, "axes.linewidth": 0.8}

# ----------------------------------------------------------------------------------
_args = [a for a in sys.argv[1:] if not a.startswith("-")]
CSV = _args[0] if _args and _args[0].lower().endswith(".csv") else CSV_PATH
OUT = _args[1] if len(_args) > 1 else OUTPUT_FOLDER
if not os.path.isfile(CSV):
    folder = os.path.dirname(os.path.abspath(CSV))
    csv_in_folder = [f for f in os.listdir(folder) if f.lower().endswith(".csv")] if os.path.isdir(folder) else []
    raise FileNotFoundError(
        "Could not find csv file\n"
        "  Looking for:   " + repr(os.path.abspath(CSV)) + "\n"
        "  Working folder:  " + repr(os.getcwd()) + "\n"
        "  Csv-files in searched folder: " + repr(csv_in_folder) + "\n"
        "Place the csv file in the working folder, or adjust the CSV PATH at the top of the script."
    )
os.makedirs(OUT, exist_ok=True)

d = pd.read_csv(CSV)
d = d[d["term"] != "(Intercept)"]          


def term_order(model):
    s = d[(d.model == model) & (d.outcome == SORT_BY)]
    return s.sort_values("OR", ascending=False)["term"].tolist()


def set_x(ax, lo, hi):
    ticks = [t for t in ALL_TICKS if lo * 0.8 <= t <= hi * 1.25]
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FixedFormatter([str(t) for t in ticks]))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlim(lo * 0.85, hi * 1.2)


def save(fig, fname):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUT, f"{fname}.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)


# ==================================================================================
# LAYOUT: ét panel, serif, én farve pr. udfald
# ==================================================================================
def make_apa(models, fname, setting):
    multi = len(models) > 1
    rows = []
    for k, m in enumerate(models):
        if k > 0:
            rows.append(("gap", m, ""))
        rows.append(("hdr", m, model_label(m)))
        rows += [("term", m, t) for t in term_order(m)]
        if REFERENCE.get(m):
            rows.append(("ref", m, REFERENCE[m]))
    n = len(rows)
    y_of = {i: n - 1 - i for i in range(n)}
    gap_y = [y_of[i] for i, r in enumerate(rows) if r[0] == "gap"]
    sub = d[d.model.isin(models)]
    lo, hi = sub.OR_low.min(), sub.OR_high.max()

    # Vertical displacement of the four outcomes within each row (first outcome at the top)
    step = setting["step"]
    offs = {o: ((len(OUTCOMES) - 1) / 2 - k) * step for k, o in enumerate(OUTCOMES)}

    with plt.rc_context(APA_RC):
        fig, ax = plt.subplots(figsize=(FIG_WIDTH, setting["height"] / 72))
        ax.axvline(1, color="black", lw=0.8, ls="--", zorder=1)
        band = 0
        for i, (kind, m, t) in enumerate(rows):
            y = y_of[i]
            if kind == "ref":
                ax.scatter(1, y, marker="s", s=14, color="0.55", zorder=3)
            elif kind == "term":
                if band % 2 == 0:                                        
                    ax.axhspan(y - 0.5, y + 0.5, color="0.95", lw=0, zorder=0)
                band += 1
                for o in OUTCOMES:
                    r = d[(d.model == m) & (d.outcome == o) & (d.term == t)].iloc[0]
                    yy = y + offs[o]
                    ax.hlines(yy, r.OR_low, r.OR_high, color=COLORS[o], lw=setting["lw"], zorder=2)
                    if r.p < 0.05:
                        ax.scatter(r.OR, yy, s=setting["ms"], marker=MARKERS[o], color=COLORS[o], linewidths=0, zorder=3)
                    else:
                        ax.scatter(r.OR, yy, s=setting["ms"], marker=MARKERS[o], facecolor="white",
                                   edgecolor=COLORS[o], linewidths=setting["lw"], zorder=3)
        for gy in gap_y:
            ax.axhline(gy, color="0.35", lw=1.0, zorder=1)             
        set_x(ax, lo, hi)
        x0, x1 = ax.get_xlim()
        ax.set_xlim(x0 / X_EXTRA, x1 * X_EXTRA)
        ax.set_ylim(-0.7, n - 0.3)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
        ax.set_axisbelow(True)

        ax.set_yticks([y_of[i] for i in range(n)])
        labels = []
        for kind, m, t in rows:
            if kind == "hdr":
                labels.append(t)
            elif kind == "gap":
                labels.append("")
            elif kind == "ref":
                labels.append(textwrap.fill(t + " (ref.)", LABEL_WIDTH))
            else:
                labels.append(textwrap.fill(t, LABEL_WIDTH, subsequent_indent="  ") if not multi else "  " + textwrap.fill(t, LABEL_WIDTH, subsequent_indent="  "))
        ax.set_yticklabels(labels)
        for lab, (kind, _, _) in zip(ax.get_yticklabels(), rows):
            if kind == "hdr":
                lab.set_fontweight("bold")
            elif kind == "ref":
                lab.set_style("italic")
        ax.tick_params(axis="y", length=0)

        handles = [plt.Line2D([], [], marker=MARKERS[o], ms=4.5, color=COLORS[o], lw=setting["lw"]) for o in OUTCOMES]
        ax.legend(handles, [LABELS.get(o, o) for o in OUTCOMES], loc="lower center",
                  bbox_to_anchor=(0.5, 1.01), ncol=2, frameon=False,
                  fontsize=WRITING_APA, handlelength=1.6, columnspacing=1.2)
        ax.set_xlabel("Odds ratio (log scale), 95% CI", fontsize=WRITING_APA)
        fig.tight_layout()
        if FOOTNOTE_IN_FIGURE:
            fig.text(0.5, -0.015, "Filled = p < .05; open = n.s.; grey square = reference category (OR = 1)",
                     ha="center", va="top", fontsize=WRITING_APA - 1.5, style="italic")
        save(fig, fname)


make_apa(["Model 1", "Model 2"], "figur1_model1_model2", APA_SETTING["fig1"])
make_apa(["Model 3"], "figur2_model3", APA_SETTING["fig2"])
print("Figures saved to", os.path.abspath(OUT))
